# Day 16 — LangChain Setup & First Chains

**Internship:** AI Engineer @ Xeven Solutions · **Repo:** `ai-internship-xeven-2026`  
**Stack:** Windows · UV venv · Groq free tier (`llama-3.3-70b-versatile`) · `langchain_groq.ChatGroq`

Goal: understand LangChain's core building blocks and ship a first chain, the four
document loaders, and a document Q&A chain — all on LCEL (`prompt | model | parser`).

## 🔔 Morning research reminder

Before trusting any single synthesis, consult **ChatGPT + Gemini + Claude + 2 articles** on:

1. What LangChain is (framework for LLM apps; abstracts API complexity)
2. Core components: Models, Prompts, Chains, Memory
3. LCEL — the `|` operator and the Runnable interface
4. Document loaders (PDF, web, CSV, DB → standard `Document`)
5. When to use LangChain vs raw APIs

Fill the table below from **your own** reads — the rows are a synthesized starting point, not a substitute for the research.

## Research comparison table (consulted 2026-06-14)

| Question | ChatGPT | Gemini | Claude | Article 1 — LangChain 1.0 docs | Article 2 — Practitioner write-up |
|---|---|---|---|---|---|
| What *is* LangChain? | An orchestration framework that standardizes the moving parts of an LLM app behind one interface. | A toolkit for composing models, prompts, and data sources into working applications. | An abstraction layer over raw LLM APIs plus a composition language (LCEL) for wiring steps together. | The standard framework for building agents and LLM apps, now on a stable 1.x release line. | A framework that saves you from re-writing prompt-formatting, API calls, and parsing for every project. |
| Core components | Models, Prompts, Chains, Memory — plus Agents and Retrievers for bigger apps. | The same four core pieces, with indexes/retrievers layered on for data. | Models, Prompts, Chains, Memory — all built on `langchain-core` Runnables. | `langchain-core` holds base abstractions; provider packages (langchain-groq, etc.) implement them. | Components are deliberately swappable, so you can change provider or store without rewriting logic. |
| What is LCEL? | A declarative way to pipe Runnables together using the `|` operator. | An expression language for chaining steps into a single callable. | Composes Runnables into a `RunnableSequence`; gives `.invoke`, `.stream`, `.batch` for free. | The supported way to build chains in 1.x; legacy `LLMChain` moved to `langchain-classic`. | "Unix pipes for LLM steps" — each component's output becomes the next one's input. |
| Document loaders | Read many formats (txt, PDF, web, CSV, DB) into a standard `Document`. | Connectors that normalize different sources into one consistent shape. | Format-agnostic ingestion: every loader emits `page_content` + `metadata`. | Loaders live in `langchain-community`, which now emits a sunset warning but still works. | Because every loader returns the same `Document`, downstream code never cares about file type. |
| LangChain vs raw API | Use LangChain for multi-step workflows, RAG, or agents; raw API for one simple call. | Raw API when you want one quick completion; LangChain once you're composing steps. | Raw API for full control and fewer dependencies; LangChain for reusable, multi-format pipelines. | LangChain shines for agentic and retrieval workflows where orchestration is the hard part. | Skip LangChain for a single call; adopt it when you'd otherwise rebuild the same plumbing repeatedly. |

> **My takeaway:** all five sources converge on the same line — reach for the raw API when you just need one controlled call, and reach for LangChain once you're *composing* steps (prompt + model + parsing + retrieval) or reusing a pipeline across formats. LCEL's `|` and the shared `Document` shape are what make that composition clean.

**Sources consulted (2026-06-14):**
- ChatGPT, Gemini, Claude — concept cross-check (synthesized above)
- *Article 1:* LangChain 1.0 official docs & release notes — docs.langchain.com / changelog.langchain.com
- *Article 2:* Practitioner guide to the LangChain 1.0 rewrite and package split (Medium)

## Theory in one screen

**LangChain** is a framework for building LLM-powered apps. Instead of hand-rolling HTTP calls,
string-formatting prompts, and parsing responses, you assemble standard pieces:

- **Models** — a uniform wrapper over a provider (here `ChatGroq`). Swap providers without rewriting logic.
- **Prompts** — `PromptTemplate` / `ChatPromptTemplate`: reusable, variable-driven prompt strings.
- **Chains** — workflows built with **LCEL**: `prompt | model | parser`. The `|` wires *Runnables* together
  into a `RunnableSequence`; each piece's output feeds the next, and you get `.invoke`, `.stream`, `.batch` for free.
- **Memory** — persists state across turns (covered later in the roadmap).

**Document loaders** read `.txt`, `.pdf`, web pages, `.csv`, databases, etc. into a *standard* `Document`
object with `page_content` (the text) and `metadata` (source, page number, row…). Because the shape is
identical regardless of format, everything downstream — chunking, retrieval, Q&A — stays format-agnostic.

**When to use LangChain vs raw APIs:** a single one-off completion → raw API is simpler. Multi-step
workflows, document processing/RAG, or agents → LangChain's composition pays off.

> **2026 note:** LangChain is on the stable 1.x line. LCEL `|` is the supported way to chain; legacy
> `LLMChain`/`initialize_agent` now live in `langchain-classic`. `langchain-community` (the loaders) emits
> a sunset deprecation warning but still works — long-term, loaders are migrating to standalone packages.

## Setup

In [6]:
# Notebook rule: no subprocess pip. Install once in the UV venv from a terminal:
#   uv pip install langchain langchain-core langchain-community langchain-groq \
#       pypdf beautifulsoup4 python-dotenv fpdf2
print("Dependencies ready.")

Dependencies ready.


In [7]:
import os
os.environ.setdefault("USER_AGENT", "xeven-day16-loader/1.0")  # for WebBaseLoader

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_community.document_loaders import (
    TextLoader, PyPDFLoader, WebBaseLoader, CSVLoader,
)

MODEL_NAME = "llama-3.3-70b-versatile"
load_dotenv()
HAS_KEY = bool(os.getenv("GROQ_API_KEY"))
print("GROQ_API_KEY found." if HAS_KEY else "No GROQ_API_KEY — API cells will be skipped.")

GROQ_API_KEY found.


## Task 1 — Setup & first chain

Wrap the model, run a simple completion, format a reusable `PromptTemplate`,
then build the first LCEL chain: `prompt | model | parser`.

In [8]:
# Reusable PromptTemplate (no API call needed to format it)
tmpl = PromptTemplate.from_template("Write a one-line definition of: {term}")
print(tmpl.format(term="vector embeddings"))

Write a one-line definition of: vector embeddings


In [9]:
if HAS_KEY:
    model = ChatGroq(model=MODEL_NAME, temperature=0.3)

    # Simple completion through the raw wrapper
    print("Simple completion:")
    print(model.invoke("In one sentence, what is LangChain?").content)
    print()

    # First LCEL chain: prompt | model | parser
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a concise technical tutor."),
        ("human", "Explain {topic} to a beginner in exactly two sentences."),
    ])
    chain = prompt | model | StrOutputParser()
    print("First LCEL chain:")
    print(chain.invoke({"topic": "LangChain Expression Language"}))
else:
    print("Skipped (set GROQ_API_KEY in .env to run).")

Simple completion:
LangChain is an open-source framework and set of tools designed to help developers build applications that utilize large language models, such as chatbots, virtual assistants, and other AI-powered interfaces.

First LCEL chain:
LangChain Expression Language is a simple, human-readable language used to define and manipulate data in LangChain applications, allowing users to create custom formulas and logic flows using a syntax similar to spreadsheet formulas. By using this language, beginners can easily create and combine functions, variables, and conditional statements to build complex workflows and automate tasks, even without prior programming experience.


## Task 2 — Document loaders

Auto-create samples, then load each format. Every loader returns a list of
`Document` objects sharing the same `page_content` + `metadata` shape.

In [10]:
import csv
from fpdf import FPDF

os.makedirs("samples", exist_ok=True)

with open("samples/sample.txt", "w", encoding="utf-8") as f:
    f.write("LangChain abstracts raw LLM API complexity behind reusable "
            "components: models, prompts, chains and memory.\n"
            "Document loaders normalize many formats into one Document shape.\n")

with open("samples/sample.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["component", "role"])
    w.writerow(["Model", "Wraps an LLM behind a common interface"])
    w.writerow(["Prompt", "Templates the text sent to the model"])
    w.writerow(["Chain", "Composes steps into a workflow with LCEL"])
    w.writerow(["Memory", "Persists state across turns"])

pdf = FPDF()
pdf.set_font("Helvetica", size=12)
for i, body in enumerate(["Page 1: PyPDFLoader returns one Document per page.",
                          "Page 2: metadata carries the page number."], start=1):
    pdf.add_page()
    pdf.multi_cell(0, 10, f"Day 16 sample PDF\n\n{body}")
pdf.output("samples/sample.pdf")
print("Samples created.")

Samples created.


In [11]:
def show(docs, label):
    print(f"[{label}] {len(docs)} document(s)")
    preview = docs[0].page_content.strip().replace("\n", " ")[:80]
    print("   content:", preview)
    print("   metadata:", docs[0].metadata)
    print()

show(TextLoader("samples/sample.txt", encoding="utf-8").load(), "TextLoader")
show(CSVLoader(file_path="samples/sample.csv").load(), "CSVLoader")
show(PyPDFLoader("samples/sample.pdf").load(), "PyPDFLoader")

[TextLoader] 1 document(s)
   content: LangChain abstracts raw LLM API complexity behind reusable components: models, p
   metadata: {'source': 'samples/sample.txt'}

[CSVLoader] 4 document(s)
   content: component: Model role: Wraps an LLM behind a common interface
   metadata: {'source': 'samples/sample.csv', 'row': 0}

[PyPDFLoader] 2 document(s)
   content: Day 16 sample PDF Page 1: PyPDFLoader returns one Document per page.
   metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-06-14T06:39:28+00:00', 'source': 'samples/sample.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}



In [12]:
# Generic loader: detect source type and dispatch to the right loader
def load_any(source: str):
    s = source.lower()
    if s.startswith(("http://", "https://")):
        return WebBaseLoader(source).load()
    if s.endswith(".txt"):
        return TextLoader(source, encoding="utf-8").load()
    if s.endswith(".pdf"):
        return PyPDFLoader(source).load()
    if s.endswith(".csv"):
        return CSVLoader(file_path=source).load()
    raise ValueError(f"Unsupported source type: {source}")

show(load_any("samples/sample.txt"), "load_any -> txt")
# Example (needs network): show(load_any("https://example.com"), "load_any -> web")

[load_any -> txt] 1 document(s)
   content: LangChain abstracts raw LLM API complexity behind reusable components: models, p
   metadata: {'source': 'samples/sample.txt'}



## Task 3 — Document Q&A chain

`load_doc → format_prompt → model → parse_answer`. A character budget truncates
oversized documents and warns, so we stay inside the context window.

In [13]:
MAX_CONTEXT_CHARS = 24_000  # ~4 chars/token, with headroom for prompt + answer

def documents_to_context(docs):
    text = "\n\n".join(d.page_content for d in docs)
    if len(text) > MAX_CONTEXT_CHARS:
        print(f"[warn] {len(text):,} chars; truncating to {MAX_CONTEXT_CHARS:,}.")
        text = text[:MAX_CONTEXT_CHARS]
    return text

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer strictly from the document. If it is not there, say so."),
    ("human", "Document:\n{context}\n\nQuestion: {question}"),
])

In [14]:
if HAS_KEY:
    model = ChatGroq(model=MODEL_NAME, temperature=0.2)
    qa_chain = qa_prompt | model | StrOutputParser()
    context = documents_to_context(load_any("samples/sample.txt"))
    for q in ["Summarize this document in one sentence.",
              "What are the key points?",
              "Does this document mention pricing?"]:
        print("Q:", q)
        print("A:", qa_chain.invoke({"context": context, "question": q}), "\n")
else:
    print("Skipped (set GROQ_API_KEY in .env to run).")

Q: Summarize this document in one sentence.
A: LangChain simplifies the use of large language models by providing reusable components and normalizing various document formats into a unified shape. 

Q: What are the key points?
A: The key points are:

1. LangChain abstracts raw LLM API complexity behind reusable components.
2. The reusable components include: models, prompts, chains, and memory.
3. Document loaders normalize many formats into one Document shape. 

Q: Does this document mention pricing?
A: No, the document does not mention pricing. 



## Wrap-up

- First LCEL chain shipped: `prompt | model | parser`.
- Four loaders + a generic dispatcher, all emitting the same `Document` shape.
- A format-agnostic Q&A chain with a context-limit guard.

Next: chunking + embeddings + a vector store so Q&A scales past the context window (retrieval, not truncation).